In [2]:
import urllib.request

urllib.request.urlretrieve(
    "https://www.gutenberg.org/cache/epub/2000/pg2000.txt", "input_raw.txt"
)

text = open("input_raw.txt", encoding="utf-8").read()
start, end = text.find("*** START"), text.find("*** END")
text = text[text.find("\n", start) + 1 : end].strip()

with open("input.txt", "w", encoding="utf-8") as f:
    f.write(text)

print(len(text), "caracteres")

2110726 caracteres


In [3]:
print(text[:1000])

El ingenioso hidalgo don Quijote de la Mancha



por Miguel de Cervantes Saavedra





El ingenioso hidalgo don Quijote de la Mancha


  
Tasa

  
Testimonio de las erratas

  
El Rey

  
Al Duque de Béjar

  
Prólogo

  
Al libro de don Quijote de la Mancha



Que trata de la condición y ejercicio del famoso
hidalgo don Quijote de la Mancha

Que trata de la primera salida que de su tierra hizo
el ingenioso don Quijote

Donde se cuenta la graciosa manera que tuvo don
Quijote en armarse caballero

De lo que le sucedió a nuestro caballero cuando salió
de la venta

Donde se prosigue la narración de la desgracia de
nuestro caballero

Del donoso y grande escrutinio que el cura y el
barbero hicieron en la librería de nuestro ingenioso hidalgo

De la segunda salida de nuestro buen caballero don
Quijote de la Mancha

Del buen suceso que el valeroso don Quijote tuvo en
la espantable y jamás imaginada aventura de los molinos de viento, con
otros sucesos dignos de felice recordación

Donde se con

In [4]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !"'(),-.01234567:;?ABCDEFGHIJLMNOPQRSTUVWXYZ]abcdefghijlmnopqrstuvxyz¡«»¿ÁÉÍÑÓÚàáéíïñóùúü—
92


In [6]:
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join(itos[i] for i in l)

print (encode("Hello there"))
print(decode(encode("Hello there")))

[28, 51, 57, 57, 60, 1, 65, 54, 51, 63, 51]
Hello there


In [9]:
%pip install torch

   ---------------------------------------- 0.0/124.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/124.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/124.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/124.1 MB ? eta -:--:--
   ---------------------------------------- 0.3/124.1 MB ? eta -:--:--
   ---------------------------------------- 0.3/124.1 MB ? eta -:--:--
   ---------------------------------------- 0.5/124.1 MB 566.4 kB/s eta 0:03:39
   ---------------------------------------- 0.5/124.1 MB 566.4 kB/s eta 0:03:39
   ---------------------------------------- 0.8/124.1 MB 590.8 kB/s eta 0:03:29
   ---------------------------------------- 0.8/124.1 MB 590.8 kB/s eta 0:03:29
   ---------------------------------------- 1.0/124.1 MB 645.6 kB/s eta 0:03:11
   ---------------------------------------- 1.3/124.1 MB 673.3 kB/s eta 0:03:03
   ---------------------------------------- 1.3/124.1 MB 673.3 kB/s eta 0:03:03
    ----------


[notice] A new release of pip is available: 24.3.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import torch
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000])

torch.Size([2110726]) torch.int64
tensor([25, 57,  1, 55, 59, 53, 51, 59, 55, 60, 64, 60,  1, 54, 55, 50, 47, 57,
        53, 60,  1, 50, 60, 59,  1, 36, 66, 55, 56, 60, 65, 51,  1, 50, 51,  1,
        57, 47,  1, 32, 47, 59, 49, 54, 47,  0,  0,  0,  0, 61, 60, 63,  1, 32,
        55, 53, 66, 51, 57,  1, 50, 51,  1, 23, 51, 63, 67, 47, 59, 65, 51, 64,
         1, 38, 47, 47, 67, 51, 50, 63, 47,  0,  0,  0,  0,  0,  0, 25, 57,  1,
        55, 59, 53, 51, 59, 55, 60, 64, 60,  1, 54, 55, 50, 47, 57, 53, 60,  1,
        50, 60, 59,  1, 36, 66, 55, 56, 60, 65, 51,  1, 50, 51,  1, 57, 47,  1,
        32, 47, 59, 49, 54, 47,  0,  0,  0,  1,  1,  0, 39, 47, 64, 47,  0,  0,
         1,  1,  0, 39, 51, 64, 65, 55, 58, 60, 59, 55, 60,  1, 50, 51,  1, 57,
        47, 64,  1, 51, 63, 63, 47, 65, 47, 64,  0,  0,  1,  1,  0, 25, 57,  1,
        37, 51, 69,  0,  0,  1,  1,  0, 21, 57,  1, 24, 66, 62, 66, 51,  1, 50,
        51,  1, 22, 83, 56, 47, 63,  0,  0,  1,  1,  0, 35, 63, 87, 57, 60, 53,
      

In [11]:
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [12]:
block_size = 8
train_data[:block_size+1]

tensor([25, 57,  1, 55, 59, 53, 51, 59, 55])

In [13]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"When input is {context} the target: {target}")

When input is tensor([25]) the target: 57
When input is tensor([25, 57]) the target: 1
When input is tensor([25, 57,  1]) the target: 55
When input is tensor([25, 57,  1, 55]) the target: 59
When input is tensor([25, 57,  1, 55, 59]) the target: 53
When input is tensor([25, 57,  1, 55, 59, 53]) the target: 51
When input is tensor([25, 57,  1, 55, 59, 53, 51]) the target: 59
When input is tensor([25, 57,  1, 55, 59, 53, 51, 59]) the target: 55


In [15]:
torch.manual_seed(1337)
batch_size = 4
block_size = 8

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('--------')

for b in range(batch_size):
    for t in range(block_size):
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")

inputs:
torch.Size([4, 8])
tensor([[69, 47,  1, 64, 51,  1, 54, 47],
        [50, 51,  0, 50, 60, 59,  1, 23],
        [59,  1, 62, 66, 51,  1, 59, 60],
        [65, 51,  1, 50, 51,  1, 58, 55]])
targets:
torch.Size([4, 8])
tensor([[47,  1, 64, 51,  1, 54, 47, 48],
        [51,  0, 50, 60, 59,  1, 23, 55],
        [ 1, 62, 66, 51,  1, 59, 60,  1],
        [51,  1, 50, 51,  1, 58, 55,  1]])
--------
when input is [69] the target: 47
when input is [69, 47] the target: 1
when input is [69, 47, 1] the target: 64
when input is [69, 47, 1, 64] the target: 51
when input is [69, 47, 1, 64, 51] the target: 1
when input is [69, 47, 1, 64, 51, 1] the target: 54
when input is [69, 47, 1, 64, 51, 1, 54] the target: 47
when input is [69, 47, 1, 64, 51, 1, 54, 47] the target: 48
when input is [50] the target: 51
when input is [50, 51] the target: 0
when input is [50, 51, 0] the target: 50
when input is [50, 51, 0, 50] the target: 60
when input is [50, 51, 0, 50, 60] the target: 59
when input is [50, 